<a href="https://colab.research.google.com/github/Monicababu21/Text-to-Audiobook-Converter/blob/main/Text_to_Audiobook_Converter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install gradio gTTS pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 2.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 1.16.1 which is incompatible.
spacy 3.8.16 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.
transformers 5.20.1 requires huggingface-hub<3.0,>=1.31.0, but you have huggingface-hub 1.16.1 which is incompatible.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.


In [ ]:
import os
import re
import zipfile
import gradio as gr
import pandas as pd

from gtts import gTTS
from datetime import datetime

OUTPUT_DIR = "/content/audiobook_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Text-to-Audiobook Converter")
print("Libraries imported successfully!")

Text-to-Audiobook Converter
Libraries imported successfully!


In [ ]:
LANGUAGES = {
    "English": "en",
    "Tamil": "ta",
    "Hindi": "hi",
    "French": "fr",
    "Spanish": "es",
    "German": "de",
    "Japanese": "ja",
    "Italian": "it"
}

SAMPLE_TEXTS = {
    "Technology": """
    Technology plays an important role in modern life.
    Computers, smartphones, and the internet help people
    communicate, learn, and work more efficiently.
    Artificial intelligence is creating new opportunities
    in education, healthcare, and business.
    """,

    "Education": """
    Education helps people gain knowledge and develop skills.
    Students learn through reading, practical activities,
    discussions, and experiments.
    Learning is a continuous process that supports
    personal and professional development.
    """,

    "Environment": """
    Protecting the environment is everyone's responsibility.
    We can reduce waste, save water, plant trees, and use
    energy efficiently.
    Small actions can help create a cleaner and healthier
    planet for future generations.
    """
}


def load_sample(name):
    return SAMPLE_TEXTS.get(name, "")


print("Language options and sample texts are ready!")

Language options and sample texts are ready!


In [ ]:
def clean_text(text):
    text = text.replace("\r\n", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def split_into_chapters(text, max_chars=2500):
    text = clean_text(text)

    if not text:
        return []

    paragraphs = re.split(r"\n\s*\n", text)
    chapters = []
    current = ""

    for paragraph in paragraphs:
        paragraph = paragraph.strip()

        if not paragraph:
            continue

        # Split exceptionally long paragraphs into smaller pieces.
        if len(paragraph) > max_chars:
            if current:
                chapters.append(current.strip())
                current = ""

            for start in range(0, len(paragraph), max_chars):
                chapters.append(paragraph[start:start + max_chars])

            continue

        if len(current) + len(paragraph) + 2 <= max_chars:
            current = (current + "\n\n" + paragraph).strip()
        else:
            if current:
                chapters.append(current.strip())
            current = paragraph

    if current:
        chapters.append(current.strip())

    return chapters


def get_text_statistics(text):
    words = re.findall(r"\b[\w'-]+\b", text)
    sentences = [
        item for item in re.split(r"[.!?]+", text)
        if item.strip()
    ]

    chapters = split_into_chapters(text)

    return {
        "Characters": len(text),
        "Words": len(words),
        "Sentences": len(sentences),
        "Estimated Chapters": len(chapters)
    }


print("Text processing functions are ready!")

Text processing functions are ready!


In [ ]:
def generate_audiobook(text, language, speed):
    text = clean_text(text)

    if not text:
        raise gr.Error("Please enter text or upload a text file.")

    if len(text) > 100000:
        raise gr.Error(
            "This prototype supports up to 100,000 characters per request."
        )

    chapters = split_into_chapters(text)

    if not chapters:
        raise gr.Error("No readable text was found.")

    language_code = LANGUAGES[language]
    slow = speed == "Slow"

    chapter_files = []
    chapter_rows = []

    for index, chapter_text in enumerate(chapters, start=1):
        chapter_path = os.path.join(
            OUTPUT_DIR,
            f"chapter_{index:03d}.mp3"
        )

        try:
            tts = gTTS(
                text=chapter_text,
                lang=language_code,
                slow=slow
            )
            tts.save(chapter_path)

        except Exception as error:
            raise gr.Error(
                "Audio generation failed. Check your internet connection "
                "and confirm that the selected language is supported."
            )

        chapter_files.append(chapter_path)

        chapter_rows.append({
            "Chapter": f"Chapter {index}",
            "Words": len(chapter_text.split()),
            "Characters": len(chapter_text),
            "Audio File": os.path.basename(chapter_path)
        })

    # Create a ZIP file containing the individual chapter audio files.
    zip_path = os.path.join(OUTPUT_DIR, "complete_audiobook.zip")

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for chapter_path in chapter_files:
            archive.write(
                chapter_path,
                arcname=os.path.basename(chapter_path)
            )

    # Create a single continuous MP3 by joining chapter audio bytes.
    # gTTS produces MP3 streams, but byte concatenation may not play
    # consistently in every audio player. The ZIP is the reliable
    # download containing separate chapter files.
    combined_path = None

    if len(chapter_files) == 1:
        combined_path = chapter_files[0]

    chapter_df = pd.DataFrame(chapter_rows)

    return (
        chapter_files[0],
        zip_path,
        chapter_df,
        combined_path,
        len(chapters)
    )


print("Audiobook generation function is ready!")

Audiobook generation function is ready!


In [ ]:
def create_audiobook(text, uploaded_file, language, speed):
    if uploaded_file is not None:
        try:
            with open(uploaded_file, "r", encoding="utf-8-sig") as file:
                text = file.read()
        except Exception:
            raise gr.Error(
                "Unable to read the file. Please upload a UTF-8 TXT file."
            )

    text = clean_text(text)

    if not text:
        raise gr.Error("Please enter text or upload a TXT file.")

    stats = get_text_statistics(text)

    first_audio, zip_path, chapter_df, combined_path, chapter_count = (
        generate_audiobook(text, language, speed)
    )

    stats_df = pd.DataFrame(
        stats.items(),
        columns=["Statistic", "Value"]
    )

    report_path = os.path.join(OUTPUT_DIR, "audiobook_report.txt")

    with open(report_path, "w", encoding="utf-8") as file:
        file.write("TEXT-TO-AUDIOBOOK CONVERTER REPORT\n")
        file.write(f"Generated: {datetime.now()}\n")
        file.write(f"Language: {language}\n")
        file.write(f"Speech Speed: {speed}\n")
        file.write(f"Chapters: {chapter_count}\n\n")

        for key, value in stats.items():
            file.write(f"{key}: {value}\n")

    summary = f"""
## Audiobook Generated Successfully!

- **Language:** {language}
- **Speech speed:** {speed}
- **Total chapters:** {chapter_count}
- **Total words:** {stats["Words"]}
- **Estimated sentences:** {stats["Sentences"]}

Use the audio player to preview the first chapter.
Download the ZIP file to access all chapter audio files.
"""

    return (
        summary,
        first_audio,
        zip_path,
        chapter_df,
        stats_df,
        report_path
    )


print("Main audiobook function is ready!")

Main audiobook function is ready!


In [ ]:
CSS = """
.gradio-container {
    max-width: 1150px !important;
    margin: auto !important;
}
body {
    background: #faf8ff;
}
h1, h2, h3 {
    color: #5b21b6;
}
button.primary {
    background: #6d28d9 !important;
    color: white !important;
}
"""

with gr.Blocks(title="Text-to-Audiobook Converter") as app:

    gr.Markdown("""
    # 📚 Text-to-Audiobook Converter
    ### Turn written content into audio you can listen to anywhere.

    Enter text or upload a TXT document to generate chapter-wise audio.
    """)

    with gr.Tab("Create Audiobook"):

        with gr.Row():

            with gr.Column():
                text_input = gr.Textbox(
                    label="Enter Text",
                    placeholder="Paste your story, article, or document here...",
                    lines=12
                )

                sample_dropdown = gr.Dropdown(
                    choices=list(SAMPLE_TEXTS.keys()),
                    label="Load Sample Text"
                )

                load_button = gr.Button("Load Sample")

                file_input = gr.File(
                    label="Or Upload a TXT File",
                    file_types=[".txt"],
                    type="filepath"
                )

                language_input = gr.Dropdown(
                    choices=list(LANGUAGES.keys()),
                    value="English",
                    label="Language"
                )

                speed_input = gr.Radio(
                    choices=["Normal", "Slow"],
                    value="Normal",
                    label="Speech Speed"
                )

                generate_button = gr.Button(
                    "Generate Audiobook",
                    variant="primary"
                )

            with gr.Column():
                summary_output = gr.Markdown(
                    label="Generation Summary"
                )

                audio_output = gr.Audio(
                    label="Preview First Chapter",
                    type="filepath"
                )

                zip_output = gr.File(
                    label="Download All Chapter Audio Files"
                )

                report_output = gr.File(
                    label="Download Text Analysis Report"
                )

        gr.Markdown("## 📑 Chapter Details")

        chapter_output = gr.Dataframe(
            label="Chapter Information",
            interactive=False
        )

        gr.Markdown("## 📊 Text Statistics")

        stats_output = gr.Dataframe(
            label="Text Statistics",
            interactive=False
        )

        load_button.click(
            fn=load_sample,
            inputs=sample_dropdown,
            outputs=text_input
        )

        generate_button.click(
            fn=create_audiobook,
            inputs=[
                text_input,
                file_input,
                language_input,
                speed_input
            ],
            outputs=[
                summary_output,
                audio_output,
                zip_output,
                chapter_output,
                stats_output,
                report_output
            ]
        )

    with gr.Tab("How It Works"):
        gr.Markdown("""
        ## Conversion Process

        1. Enter text or upload a TXT document.
        2. Select the desired language.
        3. Choose normal or slow speech.
        4. The application splits the text into chapters.
        5. gTTS generates an MP3 for each chapter.
        6. The application packages the chapter audio files into a ZIP.
        7. Preview the first chapter and download all the files.

        **Note:** Audio generation requires an internet connection.
        Voice availability depends on the selected language and gTTS service.
        """)

    with gr.Tab("About"):
        gr.Markdown("""
        ## About the Project

        This application converts text into speech and organizes long text
        into manageable audio chapters.

        **Technologies Used**
        - Python
        - Google Text-to-Speech (gTTS)
        - Pandas
        - Gradio

        **Use Cases**
        - Listening to study notes
        - Converting articles into audio
        - Creating simple spoken stories
        - Supporting audio-based learning
        """)

app.launch(share=True, css=CSS)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1f93d97fc0b1beef38.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
